In [1]:

import os
import glob
import torch
import torch.nn as nn
from torchvision import models

matches = glob.glob(
    "/kaggle/input/**/resnet18_best.pth",
    recursive=True
)

if not matches:
    raise FileNotFoundError("Checkpoint not found in Kaggle inputs")

CHECKPOINT_PATH = matches[0]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)

state_dict = checkpoint["model_state_dict"]
state_dict = {
    k.removeprefix("module."): v
    for k, v in state_dict.items()
}

model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE)
model.eval()

print("MODEL LOADED SUCCESSFULLY")
print("Checkpoint:", CHECKPOINT_PATH)
print("Device:", DEVICE)
print("Best epoch:", checkpoint["epoch"])
print("Macro AUC:", checkpoint["macro_source_auc"])


MODEL LOADED SUCCESSFULLY
Checkpoint: /kaggle/input/datasets/babulkumar07/lungvision-phase2-models/lungvision_phase2_models/resnet18_best.pth
Device: cuda
Best epoch: 3
Macro AUC: 0.9499627227664532


In [2]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

def load_xray(path):
    if path.lower().endswith(".dcm"):
        ds = pydicom.dcmread(path)
        pixels = ds.pixel_array.astype(np.float32)

        try:
            pixels = apply_voi_lut(pixels, ds).astype(np.float32)
        except Exception:
            pass

        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            pixels = pixels.max() - pixels

        pixels -= pixels.min()
        if pixels.max() > 0:
            pixels /= pixels.max()

        return pixels

    return np.array(Image.open(path).convert("L"))

samples = pd.concat([
    df[(df.source == "IN-CXR") & (df.label == 0)].head(2),
    df[(df.source == "IN-CXR") & (df.label == 1)].head(2),
    df[(df.source == "VinDr") & (df.label == 1)].head(2)
]).reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(14, 10))

for ax, (_, row) in zip(axes.flat, samples.iterrows()):
    try:
        image = load_xray(row["file_path"])

        ax.imshow(image, cmap="gray")
        ax.set_title(
            f"{row['source']}\n"
            f"Actual: {row['label_name']}\n"
            f"Prediction: {'Abnormal' if row['prediction'] else 'Normal'}\n"
            f"P(Abnormal): {row['probability']:.3f}",
            fontsize=9
        )

    except Exception as e:
        ax.set_title(f"Image error: {str(e)[:60]}")

    ax.axis("off")

plt.tight_layout()
plt.show()


NameError: name 'df' is not defined

In [ ]:

import os
import pandas as pd

print("VALIDATION DATASET ACCESS CHECK")
print("-" * 45)

for source, group in df.groupby("source"):
    exists = group["file_path"].apply(os.path.exists)

    print("\nSource:", source)
    print("Total images:", len(group))
    print("Accessible:", int(exists.sum()))
    print("Missing:", int((~exists).sum()))

    if (~exists).any():
        print("Example missing path:")
        print(group.loc[~exists, "file_path"].iloc[0])

print("\nCLASS DISTRIBUTION")
print(pd.crosstab(df["source"], df["label_name"]))


In [ ]:

import re

keywords = [
    "class ", "Dataset", "transforms.Compose",
    "Resize(", "Normalize(", "ToTensor(",
    "pad", "padding", "pydicom",
    "apply_voi_lut", "MONOCHROME",
    "Image.open", "pixel_array",
    "def __getitem__", "ImageEnhance"
]

history = get_ipython().history_manager.input_hist_raw

print("SEARCHING NOTEBOOK TRAINING CODE")
print("=" * 65)

found = 0

for cell_no, code in enumerate(history):
    if not code or cell_no == 0:
        continue

    if any(word.lower() in code.lower() for word in keywords):
        print(f"\n{'='*25} CELL {cell_no} {'='*25}")
        print(code[:7000])
        found += 1

print("\nTotal matching cells:", found)


In [ ]:

comparison = samples[
    ["image_id", "source", "label", "probability", "prediction"]
].copy()

comparison["fresh_probability"] = fresh_probabilities

comparison["absolute_difference"] = np.abs(
    comparison["probability"] -
    comparison["fresh_probability"]
)

comparison["fresh_prediction"] = (
    comparison["fresh_probability"] >= 0.5
).astype(int)

comparison["prediction_matches"] = (
    comparison["prediction"].astype(int) ==
    comparison["fresh_prediction"]
)

print("FRESH INFERENCE COMPARISON")
print(comparison.to_string(index=False))

print("\nMEAN PROBABILITY DIFFERENCE:",
      comparison["absolute_difference"].mean())

print("MAX PROBABILITY DIFFERENCE:",
      comparison["absolute_difference"].max())

print("PREDICTION AGREEMENT:",
      comparison["prediction_matches"].mean() * 100, "%")


In [ ]:

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class ReproducibilityDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        tensor = prepare_tensor(row)

        return (
            tensor,
            float(row["probability"]),
            int(row["prediction"]),
            int(row["label"]),
            str(row["source"])
        )


repro_dataset = ReproducibilityDataset(df)

repro_loader = DataLoader(
    repro_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

fresh_probs = []
saved_probs = []
saved_preds = []
true_labels = []
sources = []

model.eval()

with torch.inference_mode():
    for images, old_probs, old_preds, labels, batch_sources in tqdm(
        repro_loader,
        total=len(repro_loader),
        desc="Full validation inference"
    ):
        images = images.to(DEVICE)

        logits = model(images).flatten()
        probabilities = torch.sigmoid(logits)

        fresh_probs.extend(probabilities.cpu().numpy())
        saved_probs.extend(old_probs.numpy())
        saved_preds.extend(old_preds.numpy())
        true_labels.extend(labels.numpy())
        sources.extend(list(batch_sources))

fresh_probs = np.asarray(fresh_probs)
saved_probs = np.asarray(saved_probs)
saved_preds = np.asarray(saved_preds)
true_labels = np.asarray(true_labels)

fresh_preds = (fresh_probs >= 0.5).astype(int)

differences = np.abs(fresh_probs - saved_probs)

print("\nFULL REPRODUCIBILITY RESULTS")
print("=" * 55)
print("Images tested:", len(fresh_probs))
print("Prediction agreement:",
      f"{(fresh_preds == saved_preds).mean() * 100:.4f}%")
print("Mean probability difference:", differences.mean())
print("Median probability difference:", np.median(differences))
print("Maximum probability difference:", differences.max())
print("Differences below 0.002:", 
      f"{(differences < 0.002).mean() * 100:.2f}%")

print("\nSOURCE-WISE RESULTS")

result_df = pd.DataFrame({
    "source": sources,
    "saved_probability": saved_probs,
    "fresh_probability": fresh_probs,
    "saved_prediction": saved_preds,
    "fresh_prediction": fresh_preds,
    "label": true_labels,
    "difference": differences
})

for source, group in result_df.groupby("source"):
    print("\n", source)
    print("Images:", len(group))
    print(
        "Prediction agreement:",
        f"{(group.saved_prediction == group.fresh_prediction).mean() * 100:.4f}%"
    )
    print("Mean probability difference:",
          group.difference.mean())
    print("Maximum probability difference:",
          group.difference.max())

print("\nFULL TEST COMPLETED")


In [ ]:

mismatches = result_df[
    result_df["saved_prediction"] != result_df["fresh_prediction"]
].copy()

print("MISMATCHED IMAGES:", len(mismatches))

if len(mismatches):
    print(
        mismatches[
            [
                "source",
                "label",
                "saved_probability",
                "fresh_probability",
                "saved_prediction",
                "fresh_prediction",
                "difference"
            ]
        ].to_string(index=False)
    )
else:
    print("All predictions match.")


In [ ]:

import os

roots = [
    "/kaggle/input/datasets",
    "/kaggle/input/competitions"
]

keywords = [
    "luna", "ct", "subset", "mhd",
    "raw", "segmentation", "nodule"
]

print("LUNGVISION 3D CT INPUT AUDIT")
print("=" * 55)

for root in roots:
    if not os.path.exists(root):
        continue

    print("\nROOT:", root)

    for current, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
        ]

        matched = [
            f for f in files
            if any(k in f.lower() for k in keywords)
        ]

        if matched:
            print("\nFolder:", current)
            for filename in matched[:15]:
                print(" ", filename)


In [ ]:

import os

root = "/kaggle/input/datasets"

extensions = (
    ".mhd", ".raw", ".dcm",
    ".nii", ".nii.gz", ".nrrd"
)

found = []

for current, dirs, files in os.walk(root):
    for filename in files:
        if filename.lower().endswith(extensions):
            found.append(os.path.join(current, filename))

print("TOTAL CT/IMAGE VOLUME FILES:", len(found))

for path in found[:30]:
    print(path)

print("\nFORMAT DISTRIBUTION")

from collections import Counter

print(Counter(
    os.path.splitext(path)[1].lower()
    for path in found
))


In [ ]:

import os
from pathlib import Path
from collections import Counter

ROOT = Path(
    "/kaggle/input/datasets/avc0706/luna16/"
    "seg-lungs-LUNA16/seg-lungs-LUNA16"
)

mhd_files = sorted(ROOT.rglob("*.mhd"))

print("MHD FILES:", len(mhd_files))
print("=" * 75)

results = []

for mhd in mhd_files[:20]:
    metadata = {}

    with open(mhd, "r", errors="ignore") as f:
        for line in f:
            if "=" in line:
                key, value = line.split("=", 1)
                metadata[key.strip()] = value.strip()

    raw_name = metadata.get("ElementDataFile", "")
    raw_path = mhd.parent / raw_name if raw_name else None

    results.append({
        "scan": mhd.stem,
        "dimensions": metadata.get("DimSize"),
        "spacing": metadata.get("ElementSpacing"),
        "pixel_type": metadata.get("ElementType"),
        "raw_file": raw_name,
        "raw_exists": raw_path.exists() if raw_path else False,
        "raw_size_mb": round(raw_path.stat().st_size / (1024**2), 2)
            if raw_path and raw_path.exists() else None
    })

for item in results:
    print("\nScan:", item["scan"])
    print("Dimensions:", item["dimensions"])
    print("Spacing:", item["spacing"])
    print("Pixel type:", item["pixel_type"])
    print("RAW file:", item["raw_file"])
    print("RAW exists:", item["raw_exists"])
    print("RAW size MB:", item["raw_size_mb"])

print("\n" + "=" * 75)
print("METADATA INSPECTION COMPLETE")


In [ ]:

from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/input/datasets/avc0706/luna16"),
    Path("/kaggle/input/competitions")
]

keywords = [
    "annotation", "annotations", "candidates",
    "segmentation", "mask", "label", "csv"
]

print("ANNOTATION / SEGMENTATION FILE AUDIT")
print("=" * 65)

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    print("\nROOT:", root)

    count = 0
    for path in root.rglob("*"):
        if path.is_file() and any(
            key in path.name.lower() for key in keywords
        ):
            print(path)
            count += 1

            if count >= 60:
                print("... output limited to 60 files")
                break

    if count == 0:
        print("No matching files found.")


In [ ]:

from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path(
    "/kaggle/input/datasets/avc0706/luna16/"
    "seg-lungs-LUNA16/seg-lungs-LUNA16"
)

records = []

for mhd in sorted(ROOT.glob("*.mhd")):
    metadata = {}

    with open(mhd, "r", errors="ignore") as f:
        for line in f:
            if "=" in line:
                k, v = line.split("=", 1)
                metadata[k.strip()] = v.strip()

    data_name = metadata.get("ElementDataFile", "")
    data_path = mhd.parent / data_name if data_name else None

    records.append({
        "seriesuid": mhd.stem,
        "mhd_path": str(mhd),
        "data_file": data_name,
        "data_exists": bool(data_path and data_path.exists()),
        "data_size_mb": (
            round(data_path.stat().st_size / 1024**2, 3)
            if data_path and data_path.exists() else None
        ),
        "dimensions": metadata.get("DimSize"),
        "spacing": metadata.get("ElementSpacing"),
        "element_type": metadata.get("ElementType"),
        "compressed_data": metadata.get("CompressedData"),
        "transform_matrix": metadata.get("TransformMatrix"),
        "offset": metadata.get("Offset") or metadata.get("Position")
    })

audit = pd.DataFrame(records)

print("TOTAL MHD:", len(audit))
print("DATA FILES FOUND:", audit["data_exists"].sum())
print("MISSING DATA FILES:", (~audit["data_exists"]).sum())

print("\nDATA FILE EXTENSIONS:")
print(Counter(
    Path(x).suffix.lower()
    for x in audit["data_file"]
    if x
))

print("\nELEMENT TYPES:")
print(audit["element_type"].value_counts(dropna=False))

print("\nCOMPRESSION SETTINGS:")
print(audit["compressed_data"].value_counts(dropna=False))

display(audit.head(10))

audit.to_csv(
    "/kaggle/working/lungvision_ct_integrity_audit.csv",
    index=False
)

print("\nAudit saved.")


In [ ]:

import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/avc0706/luna16/"
    "seg-lungs-LUNA16/seg-lungs-LUNA16"
)

mhd_files = sorted(ROOT.glob("*.mhd"))

# Select first CT scan
ct_path = mhd_files[0]

print("Loading CT:", ct_path.name)

# Read compressed MetaImage volume
image = sitk.ReadImage(str(ct_path))
volume = sitk.GetArrayFromImage(image)

# Metadata
spacing = image.GetSpacing()
origin = image.GetOrigin()

print("\nCT VOLUME INFORMATION")
print("=" * 50)
print("Shape (Z,Y,X):", volume.shape)
print("Spacing (X,Y,Z):", spacing)
print("Origin:", origin)
print("Voxel type:", volume.dtype)
print("Minimum:", volume.min())
print("Maximum:", volume.max())
print("Mean:", round(float(volume.mean()), 2))
print("Volume loaded:", volume.ndim == 3)

# Select central slices
z, y, x = np.array(volume.shape) // 2

# Lung window for display
def window_ct(slice_data, level=-600, width=1500):
    low = level - width / 2
    high = level + width / 2
    return np.clip(slice_data, low, high)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Axial
axes[0].imshow(
    window_ct(volume[z]),
    cmap="gray",
    origin="lower"
)
axes[0].set_title(f"Axial | Z={z}")

# Coronal
axes[1].imshow(
    window_ct(volume[:, y, :]),
    cmap="gray",
    origin="lower",
    aspect=spacing[2] / spacing[0]
)
axes[1].set_title(f"Coronal | Y={y}")

# Sagittal
axes[2].imshow(
    window_ct(volume[:, :, x]),
    cmap="gray",
    origin="lower",
    aspect=spacing[2] / spacing[1]
)
axes[2].set_title(f"Sagittal | X={x}")

for ax in axes:
    ax.axis("off")

plt.suptitle("LungVision 3D CT — Multiplanar Reconstruction")
plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
from collections import Counter

values, counts = np.unique(volume, return_counts=True)

print("UNIQUE VOXEL VALUES")
print("=" * 45)

for value, count in zip(values, counts):
    percentage = count / volume.size * 100
    print(
        f"Value {value}: {count:,} voxels "
        f"({percentage:.3f}%)"
    )

print("\nVolume shape:", volume.shape)
print("Total voxels:", f"{volume.size:,}")


In [ ]:

from pathlib import Path

root = Path("/kaggle/input/datasets/avc0706/luna16")

print("LUNA16 DIRECTORY STRUCTURE")
print("=" * 65)

for path in sorted(root.iterdir()):
    if path.is_dir():
        print("\nFOLDER:", path.name)

        for item in list(path.iterdir())[:8]:
            print("  ", item.name)
    else:
        print("FILE:", path.name)


In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/input/datasets/avc0706/luna16")

mask_root = BASE / "seg-lungs-LUNA16" / "seg-lungs-LUNA16"

# Use the scan already loaded
seriesuid = ct_path.stem

print("TARGET SERIES UID:")
print(seriesuid)

print("\nSEARCHING ORIGINAL CT SUBSETS")
print("=" * 65)

matches = []

for subset in range(10):
    folder = BASE / f"subset{subset}"

    if not folder.exists():
        continue

    for file in folder.rglob("*.mhd"):
        if file.stem == seriesuid:
            matches.append(file)

print("Matching original CT scans:", len(matches))

for path in matches:
    print("FOUND:", path)

# Inspect original CT metadata
if matches:
    original_path = matches[0]

    import SimpleITK as sitk
    original_image = sitk.ReadImage(str(original_path))
    original_volume = sitk.GetArrayFromImage(original_image)

    print("\nORIGINAL CT INFORMATION")
    print("=" * 65)
    print("Path:", original_path)
    print("Shape:", original_volume.shape)
    print("Spacing:", original_image.GetSpacing())
    print("Data type:", original_volume.dtype)
    print("Minimum HU/intensity:", original_volume.min())
    print("Maximum HU/intensity:", original_volume.max())
    print("Mean:", float(original_volume.mean()))

    print("\nMASK INFORMATION")
    print("Mask shape:", volume.shape)
    print("Mask spacing:", image.GetSpacing())

else:
    print("No matching scan found. We will inspect subset structure.")


In [ ]:

from pathlib import Path

BASE = Path("/kaggle/input/datasets/avc0706/luna16")

print("ORIGINAL LUNA16 SUBSET AUDIT")
print("=" * 60)

total = 0

for i in range(10):
    folder = BASE / f"subset{i}"

    if folder.exists():
        scans = list(folder.rglob("*.mhd"))
        raw_files = list(folder.rglob("*.raw"))

        total += len(scans)

        print(
            f"subset{i}: "
            f"{len(scans)} MHD files, "
            f"{len(raw_files)} RAW files"
        )

print("\nTOTAL ORIGINAL MHD SCANS:", total)


In [ ]:

from pathlib import Path

BASE = Path("/kaggle/input/datasets/avc0706/luna16")

seg_root = BASE / "seg-lungs-LUNA16" / "seg-lungs-LUNA16"

# Collect segmentation scan IDs
seg_ids = {
    p.stem for p in seg_root.glob("*.mhd")
}

# Collect original CT scan IDs
original_scans = {}

for i in range(10):
    folder = BASE / f"subset{i}"

    if folder.exists():
        for p in folder.rglob("*.mhd"):
            original_scans[p.stem] = str(p)

original_ids = set(original_scans)

matched = seg_ids & original_ids
missing = seg_ids - original_ids

print("SEGMENTATION SCANS:", len(seg_ids))
print("ORIGINAL CT SCANS:", len(original_ids))
print("MATCHED SCANS:", len(matched))
print("SEGMENTATION SCANS WITHOUT ORIGINAL:", len(missing))

print("\nMATCHING PERCENTAGE:",
      round(len(matched) / len(seg_ids) * 100, 2), "%")

print("\nEXAMPLE MATCHES:")
for uid in sorted(matched)[:5]:
    print(uid, "->", original_scans[uid])

print("\nEXAMPLE MISSING SCANS:")
for uid in sorted(missing)[:10]:
    print(uid)

# Check our previously selected scan
target_uid = (
    "1.3.6.1.4.1.14519.5.2.1."
    "6279.6001.100225287222365663678666836860"
)

print("\nTARGET SCAN MATCHED:", target_uid in matched)


In [ ]:

from pathlib import Path

roots = [
    Path("/kaggle/input/datasets"),
]

for root in roots:
    for path in root.iterdir():
        if path.is_dir():
            subset_counts = {}

            for i in range(10):
                candidates = list(path.rglob(f"subset{i}"))

                count = 0
                for folder in candidates:
                    if folder.is_dir():
                        count += len(list(folder.rglob("*.mhd")))

                if count:
                    subset_counts[f"subset{i}"] = count

            if subset_counts:
                print("\nDATASET:", path)
                print(subset_counts)


In [ ]:

from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/input/datasets/vafaeii")

all_mhd = list(ROOT.rglob("*.mhd"))

# Unique scans by series UID
uid_to_paths = {}

for p in all_mhd:
    uid_to_paths.setdefault(p.stem, []).append(p)

print("TOTAL MHD FILES:", len(all_mhd))
print("UNIQUE SERIES UID:", len(uid_to_paths))
print("DUPLICATE UID GROUPS:", sum(
    len(paths) > 1 for paths in uid_to_paths.values()
))

print("\nSUBSET COUNTS (UNIQUE SCANS)")

for i in range(10):
    ids = {
        p.stem for p in all_mhd
        if f"subset{i}" in p.parts
    }
    print(f"subset{i}: {len(ids)}")

print("\nEXAMPLE DATASET PATHS")
for p in all_mhd[:5]:
    print(p)


In [ ]:

from pathlib import Path
import SimpleITK as sitk

BASE = Path("/kaggle/input/datasets/avc0706/luna16")
SEG_ROOT = BASE / "seg-lungs-LUNA16" / "seg-lungs-LUNA16"
CT_ROOT = Path("/kaggle/input/datasets/vafaeii")

seg_files = {
    p.stem: p for p in SEG_ROOT.glob("*.mhd")
}

ct_files = {}

for p in CT_ROOT.rglob("*.mhd"):
    ct_files.setdefault(p.stem, p)

matched = set(seg_files) & set(ct_files)
missing = set(seg_files) - set(ct_files)

print("SEGMENTATION SCANS:", len(seg_files))
print("ORIGINAL CT UNIQUE SCANS:", len(ct_files))
print("MATCHED PAIRS:", len(matched))
print("MISSING CT SCANS:", len(missing))

print("\nPAIR MATCH RATE:",
      round(len(matched) / len(seg_files) * 100, 2), "%")

# Inspect one matching pair
if matched:
    uid = sorted(matched)[0]

    ct = sitk.ReadImage(str(ct_files[uid]))
    mask = sitk.ReadImage(str(seg_files[uid]))

    print("\nSAMPLE PAIR:", uid)
    print("CT size:", ct.GetSize())
    print("Mask size:", mask.GetSize())
    print("CT spacing:", ct.GetSpacing())
    print("Mask spacing:", mask.GetSpacing())
    print("CT origin:", ct.GetOrigin())
    print("Mask origin:", mask.GetOrigin())
    print("CT direction:", ct.GetDirection())
    print("Mask direction:", mask.GetDirection())

    print("\nGEOMETRY IDENTICAL:",
          ct.GetSize() == mask.GetSize()
          and ct.GetSpacing() == mask.GetSpacing()
          and ct.GetOrigin() == mask.GetOrigin()
          and ct.GetDirection() == mask.GetDirection())


In [ ]:

import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path("/kaggle/input/datasets/avc0706/luna16")
SEG_ROOT = BASE / "seg-lungs-LUNA16" / "seg-lungs-LUNA16"
CT_ROOT = Path("/kaggle/input/datasets/vafaeii/luna16")

# Select a matching scan
uid = "1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222365663678666836860"

ct_path = next(CT_ROOT.rglob(uid + ".mhd"))
mask_path = SEG_ROOT / (uid + ".mhd")

ct_image = sitk.ReadImage(str(ct_path))
mask_image = sitk.ReadImage(str(mask_path))

ct = sitk.GetArrayFromImage(ct_image)
mask = sitk.GetArrayFromImage(mask_image)

assert ct.shape == mask.shape
assert ct_image.GetSpacing() == mask_image.GetSpacing()
assert ct_image.GetOrigin() == mask_image.GetOrigin()
assert ct_image.GetDirection() == mask_image.GetDirection()

print("CT shape:", ct.shape)
print("CT intensity range:", ct.min(), ct.max())
print("Mask labels:", np.unique(mask))

# Central axial slice
z = ct.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original CT
axes[0].imshow(
    ct[z],
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower"
)
axes[0].set_title("Original CT")

# Segmentation
axes[1].imshow(
    mask[z],
    cmap="nipy_spectral",
    interpolation="nearest",
    origin="lower"
)
axes[1].set_title("Lung Segmentation Labels")

# Overlay
axes[2].imshow(
    ct[z],
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower"
)

overlay = np.ma.masked_where(mask[z] == 0, mask[z])

axes[2].imshow(
    overlay,
    cmap="nipy_spectral",
    alpha=0.45,
    interpolation="nearest",
    origin="lower"
)
axes[2].set_title("CT + Segmentation Overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import SimpleITK as sitk
from pathlib import Path

SEG_ROOT = Path(
    "/kaggle/input/datasets/avc0706/luna16/"
    "seg-lungs-LUNA16/seg-lungs-LUNA16"
)
CT_ROOT = Path("/kaggle/input/datasets/vafaeii/luna16")

geometry_pass = 0
geometry_fail = []

for mask_path in sorted(SEG_ROOT.glob("*.mhd")):
    uid = mask_path.stem
    ct_path = CT_ROOT / f"subset0" / uid / (uid + ".mhd")

    # Search the dataset because scans are distributed over subsets
    if not ct_path.exists():
        candidates = list(CT_ROOT.rglob(uid + ".mhd"))
        if not candidates:
            geometry_fail.append((uid, "CT missing"))
            continue
        ct_path = candidates[0]

    ct_img = sitk.ReadImage(str(ct_path))
    mask_img = sitk.ReadImage(str(mask_path))

    same = (
        ct_img.GetSize() == mask_img.GetSize()
        and np.allclose(ct_img.GetSpacing(), mask_img.GetSpacing())
        and np.allclose(ct_img.GetOrigin(), mask_img.GetOrigin())
        and np.allclose(ct_img.GetDirection(), mask_img.GetDirection())
    )

    if same:
        geometry_pass += 1
    else:
        geometry_fail.append((uid, "Geometry mismatch"))

print("GEOMETRY PASSED:", geometry_pass)
print("GEOMETRY FAILED:", len(geometry_fail))

if geometry_fail:
    print("Examples:", geometry_fail[:10])


In [ ]:

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("skimage") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "scikit-image"
    ])

import numpy as np
import plotly.graph_objects as go
from skimage.measure import marching_cubes

# Use separate surfaces for the two main lung labels
mask_array = sitk.GetArrayFromImage(mask_image)

spacing_x, spacing_y, spacing_z = ct_image.GetSpacing()
origin = np.array(ct_image.GetOrigin())
direction = np.array(ct_image.GetDirection()).reshape(3, 3)

STEP = 2

fig = go.Figure()

lung_labels = [
    (3, "Lung label 3", "#36A2EB"),
    (4, "Lung label 4", "#FFB020")
]

for label, name, color in lung_labels:

    binary = (mask_array[::STEP, ::STEP, ::STEP] == label)

    if not binary.any():
        print("Skipping empty label:", label)
        continue

    vertices, faces, normals, values = marching_cubes(
        binary.astype(np.uint8),
        level=0.5,
        spacing=(
            spacing_z * STEP,
            spacing_y * STEP,
            spacing_x * STEP
        )
    )

    # Convert Z,Y,X coordinates into X,Y,Z physical coordinates
    xyz = vertices[:, [2, 1, 0]]

    # Apply image direction and physical origin
    physical_xyz = xyz @ direction.T + origin

    fig.add_trace(go.Mesh3d(
        x=physical_xyz[:, 0],
        y=physical_xyz[:, 1],
        z=physical_xyz[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color,
        opacity=0.85,
        name=name,
        hoverinfo="name"
    ))

fig.update_layout(
    title="LungVision AI — Interactive 3D Lung Reconstruction",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data"
    ),
    width=1000,
    height=750
)

fig.show()

output_path = "/kaggle/working/lungvision_3d_lung_model.html"
fig.write_html(output_path)

print("\n3D MODEL EXPORTED:", output_path)
print("Surface labels: 3 and 4")
print("Voxel downsampling factor:", STEP)
print("Label 5 excluded pending anatomical verification.")


In [ ]:

import os

print("HTML exists:", os.path.exists(output_path))
print("HTML size (MB):", round(os.path.getsize(output_path) / 1024**2, 2))

print("\n3D MODEL TRACE INFORMATION")

for trace in fig.data:
    print("Name:", trace.name)
    print("Vertices:", len(trace.x))
    print("Triangles:", len(trace.i))


In [ ]:

from IPython.display import display, HTML

fig.show(
    renderer="iframe",
    config={
        "responsive": True,
        "displaylogo": False,
        "scrollZoom": True
    }
)


In [ ]:

from IPython.display import IFrame, display

display(
    IFrame(
        src=output_path,
        width="1000",
        height="750"
    )
)


In [ ]:

import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/input/datasets/avc0706/luna16")

annotations = pd.read_csv(BASE / "annotations.csv")

uid = ct_path.stem

scan_annotations = annotations[
    annotations["seriesuid"] == uid
].copy()

print("ANNOTATION DATASET SHAPE:", annotations.shape)
print("ANNOTATION COLUMNS:", annotations.columns.tolist())
print("ANNOTATIONS FOR SELECTED SCAN:", len(scan_annotations))

display(scan_annotations)


In [ ]:

import numpy as np
import SimpleITK as sitk

mapped_annotations = []

for _, row in scan_annotations.iterrows():

    physical_point = (
        float(row["coordX"]),
        float(row["coordY"]),
        float(row["coordZ"])
    )

    voxel_index = ct_image.TransformPhysicalPointToIndex(
        physical_point
    )

    x, y, z = voxel_index

    inside = (
        0 <= x < ct_image.GetSize()[0]
        and 0 <= y < ct_image.GetSize()[1]
        and 0 <= z < ct_image.GetSize()[2]
    )

    mapped_annotations.append({
        "seriesuid": uid,
        "coordX_mm": physical_point[0],
        "coordY_mm": physical_point[1],
        "coordZ_mm": physical_point[2],
        "voxelX": x,
        "voxelY": y,
        "voxelZ": z,
        "diameter_mm": float(row["diameter_mm"]),
        "inside_volume": inside
    })

mapped_df = pd.DataFrame(mapped_annotations)

print("MAPPED NODULES:", len(mapped_df))

if len(mapped_df):
    print("Inside CT volume:", mapped_df["inside_volume"].sum())
    display(mapped_df)


In [ ]:

valid = mapped_df[mapped_df["inside_volume"]].copy()

if len(valid):
    fig.add_trace(go.Scatter3d(
        x=valid["coordX_mm"],
        y=valid["coordY_mm"],
        z=valid["coordZ_mm"],
        mode="markers",
        name="LUNA16 annotated nodules",
        marker=dict(
            size=7,
            color="red",
            symbol="circle",
            line=dict(color="white", width=1)
        ),
        text=[
            f"Diameter: {d:.2f} mm"
            for d in valid["diameter_mm"]
        ],
        hovertemplate="%{text}<extra></extra>"
    ))

fig.update_layout(
    title="LungVision AI — 3D Lungs with Nodule Annotations"
)

fig.show()


In [ ]:

from pathlib import Path
import pandas as pd
import SimpleITK as sitk
import numpy as np

BASE = Path("/kaggle/input/datasets/avc0706/luna16")
CT_ROOT = Path("/kaggle/input/datasets/vafaeii/luna16")
SEG_ROOT = BASE / "seg-lungs-LUNA16" / "seg-lungs-LUNA16"

annotations = pd.read_csv(BASE / "annotations.csv")

# Build unique CT and mask indexes
ct_files = {
    p.stem: p for p in CT_ROOT.rglob("*.mhd")
}

mask_files = {
    p.stem: p for p in SEG_ROOT.glob("*.mhd")
}

# Find annotated scans with both CT and segmentation
annotation_counts = annotations.groupby("seriesuid").size()

eligible = [
    uid for uid in annotation_counts.index
    if uid in ct_files and uid in mask_files
]

print("Annotated scan IDs:", len(annotation_counts))
print("Eligible CT + mask + annotation scans:", len(eligible))

# Select a scan with the most annotations
uid = max(eligible, key=lambda x: annotation_counts[x])

print("\nSELECTED SERIES UID:")
print(uid)
print("Number of annotations:", annotation_counts[uid])

# Load the exact matching pair
ct_path = ct_files[uid]
mask_path = mask_files[uid]

ct_image = sitk.ReadImage(str(ct_path))
mask_image = sitk.ReadImage(str(mask_path))

ct = sitk.GetArrayFromImage(ct_image)
mask = sitk.GetArrayFromImage(mask_image)

scan_annotations = annotations[
    annotations["seriesuid"] == uid
].copy()

print("\nCT shape:", ct.shape)
print("Mask shape:", mask.shape)
print("Annotation rows:", len(scan_annotations))

print("\nGeometry match:",
    ct_image.GetSize() == mask_image.GetSize()
    and np.allclose(ct_image.GetSpacing(), mask_image.GetSpacing())
    and np.allclose(ct_image.GetOrigin(), mask_image.GetOrigin())
    and np.allclose(ct_image.GetDirection(), mask_image.GetDirection())
)

display(scan_annotations)


In [ ]:

mapped = []

for _, row in scan_annotations.iterrows():

    point_mm = (
        float(row["coordX"]),
        float(row["coordY"]),
        float(row["coordZ"])
    )

    x, y, z = ct_image.TransformPhysicalPointToIndex(point_mm)

    inside = (
        0 <= x < ct_image.GetSize()[0]
        and 0 <= y < ct_image.GetSize()[1]
        and 0 <= z < ct_image.GetSize()[2]
    )

    mapped.append({
        "seriesuid": uid,
        "x_mm": point_mm[0],
        "y_mm": point_mm[1],
        "z_mm": point_mm[2],
        "voxel_x": x,
        "voxel_y": y,
        "voxel_z": z,
        "diameter_mm": row["diameter_mm"],
        "inside_volume": inside
    })

mapped_df = pd.DataFrame(mapped)

print("TOTAL ANNOTATIONS:", len(mapped_df))
print("INSIDE CT VOLUME:", mapped_df["inside_volume"].sum())
print("OUTSIDE CT VOLUME:", (~mapped_df["inside_volume"]).sum())

display(mapped_df)


In [ ]:

import plotly.graph_objects as go

valid = mapped_df[mapped_df["inside_volume"]]

fig_nodules = go.Figure(fig.data)

if len(valid):
    fig_nodules.add_trace(go.Scatter3d(
        x=valid["x_mm"],
        y=valid["y_mm"],
        z=valid["z_mm"],
        mode="markers",
        name="Annotated pulmonary nodules",
        marker=dict(
            size=7,
            color="red",
            symbol="circle",
            line=dict(color="white", width=1)
        ),
        text=[
            f"Diameter: {d:.2f} mm"
            for d in valid["diameter_mm"]
        ],
        hovertemplate="%{text}<extra></extra>"
    ))

fig_nodules.update_layout(
    title="LungVision AI — 3D Lungs with Verified Nodule Coordinates"
)

fig_nodules.show()


In [ ]:

mask_array = sitk.GetArrayFromImage(mask_image)

label_checks = []

for _, row in mapped_df.iterrows():
    x = int(row["voxel_x"])
    y = int(row["voxel_y"])
    z = int(row["voxel_z"])

    center_label = int(mask_array[z, y, x])

    # Inspect a small neighbourhood around the nodule centre
    z0, z1 = max(0, z-2), min(mask_array.shape[0], z+3)
    y0, y1 = max(0, y-2), min(mask_array.shape[1], y+3)
    x0, x1 = max(0, x-2), min(mask_array.shape[2], x+3)

    nearby_labels, nearby_counts = np.unique(
        mask_array[z0:z1, y0:y1, x0:x1],
        return_counts=True
    )

    label_checks.append({
        "voxel": (x, y, z),
        "center_label": center_label,
        "nearby_labels": nearby_labels.tolist(),
        "inside_lung_label": center_label in (3, 4)
    })

display(pd.DataFrame(label_checks))


In [ ]:

from skimage.measure import marching_cubes
import plotly.graph_objects as go
import numpy as np

mask_array = sitk.GetArrayFromImage(mask_image)

spacing_x, spacing_y, spacing_z = ct_image.GetSpacing()
origin = np.array(ct_image.GetOrigin())
direction = np.array(ct_image.GetDirection()).reshape(3, 3)

STEP = 2

fig_verified = go.Figure()

# Reconstruct surfaces from the CURRENT scan's mask
for label, name, color in [
    (3, "Segmentation label 3", "#36A2EB"),
    (4, "Segmentation label 4", "#FFB020")
]:
    binary = (mask_array[::STEP, ::STEP, ::STEP] == label)

    if not binary.any():
        print("No voxels for:", name)
        continue

    vertices, faces, _, _ = marching_cubes(
        binary.astype(np.uint8),
        level=0.5,
        spacing=(
            spacing_z * STEP,
            spacing_y * STEP,
            spacing_x * STEP
        )
    )

    # Z,Y,X -> X,Y,Z physical coordinates
    xyz = vertices[:, [2, 1, 0]]
    physical_xyz = xyz @ direction.T + origin

    fig_verified.add_trace(go.Mesh3d(
        x=physical_xyz[:, 0],
        y=physical_xyz[:, 1],
        z=physical_xyz[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color,
        opacity=0.65,
        name=name
    ))

# Add annotations from the SAME selected scan
valid = mapped_df[mapped_df["inside_volume"]].copy()

fig_verified.add_trace(go.Scatter3d(
    x=valid["x_mm"],
    y=valid["y_mm"],
    z=valid["z_mm"],
    mode="markers",
    name="LUNA16 annotated nodules",
    marker=dict(
        size=6,
        color="red",
        symbol="circle",
        line=dict(color="white", width=1)
    ),
    text=[
        f"Nodule {i+1}<br>Diameter: {d:.2f} mm"
        for i, d in enumerate(valid["diameter_mm"])
    ],
    hovertemplate="%{text}<extra></extra>"
))

fig_verified.update_layout(
    title=f"LungVision 3D CT + Nodule Annotations",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data"
    ),
    width=1100,
    height=850
)

fig_verified.show()

print("Selected scan:", uid)
print("Lung surfaces:", len(fig_verified.data) - 1)
print("Nodule markers:", len(valid))


In [ ]:

import numpy as np
import pandas as pd

sx, sy, sz = ct_image.GetSpacing()

mask_array = sitk.GetArrayFromImage(mask_image)

results = []

for _, row in mapped_df.iterrows():

    x = int(row["voxel_x"])
    y = int(row["voxel_y"])
    z = int(row["voxel_z"])

    radius_mm = float(row["diameter_mm"]) / 2

    rx = int(np.ceil(radius_mm / sx))
    ry = int(np.ceil(radius_mm / sy))
    rz = int(np.ceil(radius_mm / sz))

    z0, z1 = max(0, z-rz), min(mask_array.shape[0], z+rz+1)
    y0, y1 = max(0, y-ry), min(mask_array.shape[1], y+ry+1)
    x0, x1 = max(0, x-rx), min(mask_array.shape[2], x+rx+1)

    zz, yy, xx = np.ogrid[z0:z1, y0:y1, x0:x1]

    distance = np.sqrt(
        ((xx-x)*sx)**2 +
        ((yy-y)*sy)**2 +
        ((zz-z)*sz)**2
    )

    sphere = distance <= radius_mm

    local_mask = mask_array[z0:z1, y0:y1, x0:x1]

    total = int(sphere.sum())
    lung_overlap = int(
        (sphere & np.isin(local_mask, [3, 4])).sum()
    )

    results.append({
        "diameter_mm": row["diameter_mm"],
        "center_label": int(mask_array[z, y, x]),
        "sphere_voxels": total,
        "lung_overlap_voxels": lung_overlap,
        "lung_overlap_percent": (
            100 * lung_overlap / total if total else 0
        )
    })

overlap_df = pd.DataFrame(results)

display(overlap_df.round(2))

print(
    "\nMean approximate lung-mask overlap:",
    round(overlap_df["lung_overlap_percent"].mean(), 2),
    "%"
)


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# Investigate nodule at table index 10
idx = 10
row = mapped_df.iloc[idx]

x = int(row["voxel_x"])
y = int(row["voxel_y"])
z = int(row["voxel_z"])

diameter = float(row["diameter_mm"])

print("NODULE 10 INVESTIGATION")
print("=" * 50)
print("Voxel:", (x, y, z))
print("Diameter:", diameter, "mm")
print("Centre label:", mask_array[z, y, x])

# Show axial, coronal and sagittal views
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Axial
axes[0].imshow(
    ct[z],
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower"
)
axes[0].contour(
    np.isin(mask_array[z], [3, 4]),
    levels=[0.5],
    colors=["lime"],
    linewidths=0.8,
    origin="lower"
)
axes[0].scatter(x, y, color="red", marker="x", s=90)
axes[0].set_title(f"Axial Z={z}")

# Coronal
axes[1].imshow(
    ct[:, y, :],
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower",
    aspect=spacing_z / spacing_x
)
axes[1].contour(
    np.isin(mask_array[:, y, :], [3, 4]),
    levels=[0.5],
    colors=["lime"],
    linewidths=0.8,
    origin="lower"
)
axes[1].scatter(x, z, color="red", marker="x", s=90)
axes[1].set_title(f"Coronal Y={y}")

# Sagittal
axes[2].imshow(
    ct[:, :, x],
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower",
    aspect=spacing_z / spacing_y
)
axes[2].contour(
    np.isin(mask_array[:, :, x], [3, 4]),
    levels=[0.5],
    colors=["lime"],
    linewidths=0.8,
    origin="lower"
)
axes[2].scatter(y, z, color="red", marker="x", s=90)
axes[2].set_title(f"Sagittal X={x}")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    "Nodule 10 — CT with Lung Segmentation Boundary\n"
    "Red = annotation centre | Green = segmentation boundary"
)
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np

idx = 10
row = mapped_df.iloc[idx]

x = int(row["voxel_x"])
y = int(row["voxel_y"])
z = int(row["voxel_z"])

diameter = float(row["diameter_mm"])

sx, sy, sz = ct_image.GetSpacing()

# Crop radius: approximately 15 mm around the nodule
rx = int(np.ceil(15 / sx))
ry = int(np.ceil(15 / sy))

x0, x1 = max(0, x-rx), min(ct.shape[2], x+rx+1)
y0, y1 = max(0, y-ry), min(ct.shape[1], y+ry+1)

ct_crop = ct[z, y0:y1, x0:x1]
mask_crop = mask_array[z, y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Original CT crop
axes[0].imshow(
    ct_crop,
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower"
)

axes[0].set_title("Original CT — Nodule 10")

# Nodule diameter shown as a circle
circle = Circle(
    (x-x0, y-y0),
    radius=diameter / (2*sx),
    edgecolor="red",
    facecolor="none",
    linewidth=2
)
axes[0].add_patch(circle)

axes[0].scatter(
    x-x0, y-y0,
    color="red",
    marker="x",
    s=80
)

# Segmentation overlay
axes[1].imshow(
    ct_crop,
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower"
)

axes[1].contour(
    np.isin(mask_crop, [3, 4]),
    levels=[0.5],
    colors=["lime"],
    linewidths=1.2,
    origin="lower"
)

axes[1].add_patch(Circle(
    (x-x0, y-y0),
    radius=diameter / (2*sx),
    edgecolor="red",
    facecolor="none",
    linewidth=2
))

axes[1].scatter(
    x-x0, y-y0,
    color="red",
    marker="x",
    s=80
)

axes[1].set_title("CT + Segmentation + Nodule Boundary")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np

idx = 10
row = mapped_df.iloc[idx]

x = int(row["voxel_x"])
y = int(row["voxel_y"])
z = int(row["voxel_z"])

radius = 15  # mm

sx, sy, sz = ct_image.GetSpacing()

rx = int(np.ceil(radius / sx))
ry = int(np.ceil(radius / sy))

crop = ct[
    z,
    max(0, y-ry):min(ct.shape[1], y+ry+1),
    max(0, x-rx):min(ct.shape[2], x+rx+1)
]

print("CT ARRAY DIAGNOSTICS")
print("=" * 45)
print("CT dtype:", ct.dtype)
print("Global min/max:", ct.min(), ct.max())
print("Crop shape:", crop.shape)
print("Crop min/max:", crop.min(), crop.max())
print("Crop mean:", crop.mean())
print("Crop standard deviation:", crop.std())
print("Crop percentiles:", np.percentile(crop, [0, 5, 25, 50, 75, 95, 100]))
print("Unique values:", len(np.unique(crop)))


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

low, high = np.percentile(crop, [2, 98])

axes[0].imshow(
    crop,
    cmap="gray",
    vmin=low,
    vmax=high,
    origin="lower"
)
axes[0].set_title(f"Adaptive CT window ({low:.0f} to {high:.0f} HU)")

axes[1].imshow(
    crop,
    cmap="gray",
    vmin=-1000,
    vmax=400,
    origin="lower"
)

mask_crop = mask_array[
    z,
    max(0, y-ry):min(mask_array.shape[1], y+ry+1),
    max(0, x-rx):min(mask_array.shape[2], x+rx+1)
]

axes[1].contour(
    np.isin(mask_crop, [3, 4]),
    levels=[0.5],
    colors="lime",
    origin="lower"
)

center_x = x - max(0, x-rx)
center_y = y - max(0, y-ry)

for ax in axes:
    ax.add_patch(Circle(
        (center_x, center_y),
        radius=float(row["diameter_mm"]) / (2*sx),
        fill=False,
        edgecolor="red",
        linewidth=2
    ))
    ax.scatter(center_x, center_y, color="red", marker="x")
    ax.axis("off")

axes[1].set_title("Standard CT window + segmentation")

plt.tight_layout()
plt.show()


In [ ]:

import SimpleITK as sitk
import numpy as np
from pathlib import Path

print("CURRENT FILE PATHS")
print("=" * 65)
print("CT path:", ct_path)
print("Mask path:", mask_path)

# Load both files independently
fresh_ct_image = sitk.ReadImage(str(ct_path))
fresh_mask_image = sitk.ReadImage(str(mask_path))

fresh_ct = sitk.GetArrayFromImage(fresh_ct_image)
fresh_mask = sitk.GetArrayFromImage(fresh_mask_image)

def inspect_volume(name, image, array):
    print("\n", name)
    print("-" * 55)
    print("Shape:", array.shape)
    print("Dtype:", array.dtype)
    print("Minimum:", array.min())
    print("Maximum:", array.max())
    print("Mean:", float(array.mean()))
    print("Standard deviation:", float(array.std()))
    print("Unique values:", len(np.unique(array)))
    print("Percentiles:",
          np.percentile(array, [0, 1, 25, 50, 75, 99, 100]))
    print("Spacing:", image.GetSpacing())
    print("Origin:", image.GetOrigin())

inspect_volume("ORIGINAL CT CANDIDATE", fresh_ct_image, fresh_ct)
inspect_volume("SEGMENTATION MASK", fresh_mask_image, fresh_mask)

print("\nARRAY COMPARISON")
print("Arrays identical:", np.array_equal(fresh_ct, fresh_mask))
print("Same unique values:",
      np.array_equal(np.unique(fresh_ct), np.unique(fresh_mask)))


In [ ]:

def inspect_mhd_header(path):
    print("\nHEADER:", path)

    with open(path, "r", errors="ignore") as f:
        for line in f:
            if any(key in line for key in [
                "ElementDataFile",
                "ElementType",
                "ElementSpacing",
                "DimSize",
                "CompressedData",
                "ElementByteOrderMSB"
            ]):
                print(line.strip())

inspect_mhd_header(ct_path)
inspect_mhd_header(mask_path)


In [ ]:

from pathlib import Path
import SimpleITK as sitk
import numpy as np

CT_ROOT = Path("/kaggle/input/datasets/vafaeii/luna16")

# Build original CT index ONLY from subset folders
original_ct_files = {}

for i in range(10):
    subset_root = CT_ROOT / f"subset{i}"

    if not subset_root.exists():
        continue

    for p in subset_root.rglob("*.mhd"):
        original_ct_files[p.stem] = p

print("ORIGINAL CT SCANS:", len(original_ct_files))

# Select the current annotated scan
uid = "1.3.6.1.4.1.14519.5.2.1.6279.6001.176030616406569931557298712518"

assert uid in original_ct_files, "Original CT not found!"

# Load genuine original CT
correct_ct_path = original_ct_files[uid]
correct_ct_image = sitk.ReadImage(str(correct_ct_path))
correct_ct = sitk.GetArrayFromImage(correct_ct_image)

# Load segmentation independently
correct_mask_path = (
    Path("/kaggle/input/datasets/avc0706/luna16/")
    / "seg-lungs-LUNA16/seg-lungs-LUNA16"
    / f"{uid}.mhd"
)

correct_mask_image = sitk.ReadImage(str(correct_mask_path))
correct_mask = sitk.GetArrayFromImage(correct_mask_image)

print("\nCORRECT ORIGINAL CT")
print("Path:", correct_ct_path)
print("Shape:", correct_ct.shape)
print("Dtype:", correct_ct.dtype)
print("Min:", correct_ct.min())
print("Max:", correct_ct.max())
print("Mean:", correct_ct.mean())
print("Unique values:", len(np.unique(correct_ct)))
print("Percentiles:", np.percentile(
    correct_ct, [0, 1, 5, 25, 50, 75, 95, 99, 100]
))

print("\nSEGMENTATION MASK")
print("Min:", correct_mask.min())
print("Max:", correct_mask.max())
print("Unique labels:", np.unique(correct_mask))

print("\nGEOMETRY MATCH:",
    correct_ct_image.GetSize() == correct_mask_image.GetSize()
    and np.allclose(correct_ct_image.GetSpacing(), correct_mask_image.GetSpacing())
    and np.allclose(correct_ct_image.GetOrigin(), correct_mask_image.GetOrigin())
    and np.allclose(correct_ct_image.GetDirection(), correct_mask_image.GetDirection())
)


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Nodule 10 coordinates from the verified annotation table
row = mapped_df.iloc[10]

x = int(row["voxel_x"])
y = int(row["voxel_y"])
z = int(row["voxel_z"])

diameter = float(row["diameter_mm"])

sx, sy, sz = correct_ct_image.GetSpacing()

radius_x = int(np.ceil(15 / sx))
radius_y = int(np.ceil(15 / sy))

x0, x1 = max(0, x-radius_x), min(correct_ct.shape[2], x+radius_x+1)
y0, y1 = max(0, y-radius_y), min(correct_ct.shape[1], y+radius_y+1)

ct_crop = correct_ct[z, y0:y1, x0:x1]
mask_crop = correct_mask[z, y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].imshow(
    ct_crop, cmap="gray",
    vmin=-1000, vmax=400, origin="lower"
)
axes[0].set_title("Original LUNA16 CT")

axes[1].imshow(
    ct_crop, cmap="gray",
    vmin=-1000, vmax=400, origin="lower"
)
axes[1].contour(
    np.isin(mask_crop, [3, 4]),
    levels=[0.5],
    colors=["lime"],
    origin="lower"
)
axes[1].set_title("Original CT + Lung Mask")

cx, cy = x-x0, y-y0

for ax in axes:
    ax.add_patch(Circle(
        (cx, cy),
        diameter/(2*sx),
        fill=False,
        edgecolor="red",
        linewidth=2
    ))
    ax.scatter(cx, cy, color="red", marker="x")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np

idx = 10
row = mapped_df.iloc[idx]

x = int(row["voxel_x"])
y = int(row["voxel_y"])
z = int(row["voxel_z"])

diameter = float(row["diameter_mm"])

sx, sy, sz = correct_ct_image.GetSpacing()

rx = int(np.ceil(15 / sx))
ry = int(np.ceil(15 / sy))

x0 = max(0, x-rx)
x1 = min(correct_ct.shape[2], x+rx+1)
y0 = max(0, y-ry)
y1 = min(correct_ct.shape[1], y+ry+1)

slices = range(max(0, z-3), min(correct_ct.shape[0], z+4))

fig, axes = plt.subplots(1, len(slices), figsize=(19, 4))

for ax, slice_z in zip(axes, slices):

    ct_crop = correct_ct[
        slice_z, y0:y1, x0:x1
    ]

    mask_crop = correct_mask[
        slice_z, y0:y1, x0:x1
    ]

    ax.imshow(
        ct_crop,
        cmap="gray",
        vmin=-1000,
        vmax=400,
        origin="lower"
    )

    ax.contour(
        np.isin(mask_crop, [3, 4]),
        levels=[0.5],
        colors=["lime"],
        linewidths=0.8,
        origin="lower"
    )

    ax.add_patch(Circle(
        (x-x0, y-y0),
        radius=diameter/(2*sx),
        fill=False,
        edgecolor="red",
        linewidth=1.5
    ))

    ax.scatter(
        x-x0, y-y0,
        color="red",
        marker="x",
        s=35
    )

    ax.set_title(f"Z = {slice_z}")
    ax.axis("off")

plt.suptitle(
    "Nodule 10 — Adjacent Axial CT Slices\n"
    "Red = annotation | Green = lung segmentation boundary"
)

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import pandas as pd

idx = 10
row = mapped_df.iloc[idx]

x = int(row["voxel_x"])
y = int(row["voxel_y"])
z = int(row["voxel_z"])

radius_mm = float(row["diameter_mm"]) / 2

sx, sy, sz = correct_ct_image.GetSpacing()

rx = int(np.ceil(radius_mm / sx))
ry = int(np.ceil(radius_mm / sy))
rz = int(np.ceil(radius_mm / sz))

slice_results = []

for zz in range(max(0, z-rz), min(correct_ct.shape[0], z+rz+1)):

    z_distance = (zz-z)*sz

    remaining_radius_sq = radius_mm**2 - z_distance**2

    if remaining_radius_sq < 0:
        continue

    slice_radius = np.sqrt(remaining_radius_sq)

    y0 = max(0, y-ry)
    y1 = min(correct_ct.shape[1], y+ry+1)
    x0 = max(0, x-rx)
    x1 = min(correct_ct.shape[2], x+rx+1)

    yy, xx = np.ogrid[y0:y1, x0:x1]

    distance_sq = (
        ((xx-x)*sx)**2 +
        ((yy-y)*sy)**2
    )

    sphere_slice = distance_sq <= remaining_radius_sq

    mask_slice = correct_mask[zz, y0:y1, x0:x1]

    lung_pixels = np.isin(mask_slice, [3, 4])

    total = int(sphere_slice.sum())
    overlap = int((sphere_slice & lung_pixels).sum())

    slice_results.append({
        "z_slice": zz,
        "axial_distance_mm": round(z_distance, 2),
        "sphere_pixels": total,
        "lung_overlap_pixels": overlap,
        "overlap_percent": round(
            100*overlap/total if total else 0, 2
        )
    })

slice_df = pd.DataFrame(slice_results)

display(slice_df)

print(
    "Volume-weighted overlap:",
    round(
        100 * slice_df["lung_overlap_pixels"].sum()
        / slice_df["sphere_pixels"].sum(),
        2
    ),
    "%"
)


In [ ]:

import json
from pathlib import Path
import pandas as pd
import numpy as np

audit = {
    "project": "LungVision AI",
    "dataset": "LUNA16",
    "seriesuid": uid,
    "original_ct_path": str(correct_ct_path),
    "segmentation_path": str(correct_mask_path),
    "ct_shape_zyx": list(correct_ct.shape),
    "ct_intensity_min": int(correct_ct.min()),
    "ct_intensity_max": int(correct_ct.max()),
    "ct_unique_values": int(len(np.unique(correct_ct))),
    "geometry_match": True,
    "annotations_mapped": int(len(mapped_df)),
    "annotations_inside_ct": int(mapped_df["inside_volume"].sum()),
    "nodule_10": {
        "diameter_mm": float(mapped_df.iloc[10]["diameter_mm"]),
        "center_voxel": [
            int(mapped_df.iloc[10]["voxel_x"]),
            int(mapped_df.iloc[10]["voxel_y"]),
            int(mapped_df.iloc[10]["voxel_z"])
        ],
        "sphere_voxels": 71,
        "lung_overlap_voxels": 28,
        "lung_overlap_percent": 39.44,
        "interpretation": "Segmentation-boundary discrepancy; cause not established"
    },
    "clinical_validation": False
}

output_dir = Path("/kaggle/working/lungvision_3d_audit")
output_dir.mkdir(parents=True, exist_ok=True)

audit_path = output_dir / "nodule_annotation_audit.json"

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=4)

mapped_df.to_csv(
    output_dir / "mapped_nodule_annotations.csv",
    index=False
)

slice_df.to_csv(
    output_dir / "nodule_10_slice_overlap.csv",
    index=False
)

print("AUDIT EXPORTED")
print(audit_path)
print("Mapped annotations:", len(mapped_df))
print("Nodule 10 overlap:", 39.44, "%")


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Use verified original CT and matching mask
volume = correct_ct
lung_mask = correct_mask

# Initial position: first mapped nodule
selected = mapped_df.iloc[0]

initial_x = int(selected["voxel_x"])
initial_y = int(selected["voxel_y"])
initial_z = int(selected["voxel_z"])

# Sliders
z_slider = widgets.IntSlider(
    value=initial_z,
    min=0,
    max=volume.shape[0]-1,
    description="Axial Z",
    continuous_update=False
)

y_slider = widgets.IntSlider(
    value=initial_y,
    min=0,
    max=volume.shape[1]-1,
    description="Coronal Y",
    continuous_update=False
)

x_slider = widgets.IntSlider(
    value=initial_x,
    min=0,
    max=volume.shape[2]-1,
    description="Sagittal X",
    continuous_update=False
)

nodule_selector = widgets.Dropdown(
    options=[
        (f"Nodule {i+1} — {row.diameter_mm:.2f} mm", i)
        for i, row in mapped_df.iterrows()
    ],
    value=0,
    description="Nodule"
)

output = widgets.Output()

def update_view(z, y, x, nodule_index):

    row = mapped_df.iloc[nodule_index]

    nx = int(row["voxel_x"])
    ny = int(row["voxel_y"])
    nz = int(row["voxel_z"])

    with output:
        clear_output(wait=True)

        fig, axes = plt.subplots(1, 3, figsize=(16, 6))

        # Axial
        axes[0].imshow(
            volume[z], cmap="gray",
            vmin=-1000, vmax=400, origin="lower"
        )
        axes[0].contour(
            np.isin(lung_mask[z], [3, 4]),
            levels=[0.5], colors="lime",
            linewidths=0.6, origin="lower"
        )
        axes[0].scatter(
            nx, ny, color="red", marker="x", s=60
        )
        axes[0].axhline(y, color="cyan", linewidth=0.6)
        axes[0].axvline(x, color="cyan", linewidth=0.6)
        axes[0].set_title(f"Axial — Z={z}")

        # Coronal
        axes[1].imshow(
            volume[:, y, :], cmap="gray",
            vmin=-1000, vmax=400, origin="lower",
            aspect=sz / sx
        )
        axes[1].contour(
            np.isin(lung_mask[:, y, :], [3, 4]),
            levels=[0.5], colors="lime",
            linewidths=0.6, origin="lower"
        )
        axes[1].scatter(
            nx, nz, color="red", marker="x", s=60
        )
        axes[1].axhline(z, color="cyan", linewidth=0.6)
        axes[1].axvline(x, color="cyan", linewidth=0.6)
        axes[1].set_title(f"Coronal — Y={y}")

        # Sagittal
        axes[2].imshow(
            volume[:, :, x], cmap="gray",
            vmin=-1000, vmax=400, origin="lower",
            aspect=sz / sy
        )
        axes[2].contour(
            np.isin(lung_mask[:, :, x], [3, 4]),
            levels=[0.5], colors="lime",
            linewidths=0.6, origin="lower"
        )
        axes[2].scatter(
            ny, nz, color="red", marker="x", s=60
        )
        axes[2].axhline(z, color="cyan", linewidth=0.6)
        axes[2].axvline(y, color="cyan", linewidth=0.6)
        axes[2].set_title(f"Sagittal — X={x}")

        for ax in axes:
            ax.axis("off")

        plt.suptitle(
            f"LungVision CT Viewer | Nodule {nodule_index+1} | "
            f"Diameter: {row.diameter_mm:.2f} mm"
        )

        plt.tight_layout()
        plt.show()
        plt.close(fig)

    print(
        f"Selected nodule: {nodule_index+1} | "
        f"Voxel: ({nx}, {ny}, {nz}) | "
        f"Current slices: X={x}, Y={y}, Z={z}"
    )

controls = widgets.interactive_output(
    update_view,
    {
        "z": z_slider,
        "y": y_slider,
        "x": x_slider,
        "nodule_index": nodule_selector
    }
)

display(
    widgets.VBox([
        nodule_selector,
        z_slider,
        y_slider,
        x_slider,
        output
    ])
)

# Trigger initial render
update_view(initial_z, initial_y, initial_x, 0)


In [ ]:

def jump_to_selected_nodule(change):
    if change["name"] != "value":
        return

    row = mapped_df.iloc[change["new"]]

    # Navigate to the selected nodule's exact voxel
    z_slider.value = int(row["voxel_z"])
    y_slider.value = int(row["voxel_y"])
    x_slider.value = int(row["voxel_x"])


# Automatically navigate when dropdown changes
nodule_selector.observe(
    jump_to_selected_nodule,
    names="value"
)

print("Automatic nodule navigation enabled.")
